In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from PIL import Image
import os

# эта часть кода скопирована из readme датасета mnist
'''import requests
from io import BytesIO
from zipfile import ZipFile
DATASET_URL = 'https://github.com/DeepTrackAI/MNIST_dataset/raw/main/mnist.zip'
response = requests.get(DATASET_URL)
with ZipFile(BytesIO(response.content)) as z:
    z.extractall()'''
# дальше делал сам
input = torch.zeros((60000, 28, 28), dtype=torch.float32)
right_answer = torch.zeros(60000, dtype=torch.long)
k = 0
for img in os.listdir("mnist/train"):
    path = os.path.join("mnist/train", img)
    image = Image.open(path)
    x = [[0 for _ in range(28)] for _ in range(28)]
    for i in range(28):
        for j in range(28):
            x[i][j] = image.getpixel((j, i)) / 255
    input[k] = torch.tensor(x, dtype=torch.float32)
    right_answer[k] = int(img[0])
    k += 1
    print(k)

test = torch.zeros((10000, 28, 28), dtype=torch.float32)
test_labels = torch.zeros(10000, dtype=torch.long)
k = 0
for img in os.listdir("mnist/test"):
    path = os.path.join("mnist/test", img)
    image = Image.open(path)
    x = [[0 for _ in range(28)] for _ in range(28)]
    for i in range(28):
        for j in range(28):
            x[i][j] = image.getpixel((j, i)) / 255
    test[k] = torch.tensor(x, dtype=torch.float32)
    test_labels[k] = int(img[0])
    k += 1
    print(k)


class Mnist(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.hidden = nn.Linear(28 * 28, 120)
        self.output = nn.Linear(120, 10)

    def forward(self, input):
        input = self.flatten(input)
        input = torch.relu(self.hidden(input))
        input = self.output(input)
        return input


model = Mnist()

criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.01)

batch = 64
for epoch in range(15):
    permutation = torch.randperm(input.size(0))

    for i in range(0, input.size(0), batch):
        indices = permutation[i:i + batch]
        batch_x = input[indices]
        batch_y = right_answer[indices]

        optimizer.zero_grad()
        outputs = model(batch_x)
        loss = criterion(outputs, batch_y)
        loss.backward()
        optimizer.step()
        print(loss.item())

right = 0
with torch.no_grad():
    for k in range(len(test)):
        output = model(test[k].unsqueeze(0))
        predicted = torch.argmax(output).item()
        if predicted == test_labels[k].item():
            right += 1

accuracy = right / len(test)
print(accuracy)

ERROR:root:Internal Python error in the inspect module.
Below is the traceback from this internal error.



Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 3553, in run_code
    exec(code_obj, self.user_global_ns, self.user_ns)
  File "<ipython-input-1-f9547a847b58>", line 1, in <cell line: 0>
    import torch
  File "/usr/local/lib/python3.11/dist-packages/torch/__init__.py", line 405, in <module>
    from torch._C import *  # noqa: F403
    ^^^^^^^^^^^^^^^^^^^^^^
  File "<frozen importlib._bootstrap>", line 216, in _lock_unlock_module
KeyboardInterrupt

During handling of the above exception, another exception occurred:

Traceback (most recent call last):
  File "/usr/local/lib/python3.11/dist-packages/IPython/core/interactiveshell.py", line 2099, in showtraceback
    stb = value._render_traceback_()
          ^^^^^^^^^^^^^^^^^^^^^^^^
AttributeError: 'KeyboardInterrupt' object has no attribute '_render_traceback_'

During handling of the above exception, another exception occurred:

Traceback (most recent call last)

TypeError: object of type 'NoneType' has no len()